In [20]:
# =============================================================================
# MindGuard — Milestone 1: Data Collection & Preprocessing
# =============================================================================
#
# PURPOSE:
#   Load the 3 Kaggle datasets, clean each one individually, then merge them
#   into a single unified DataFrame ready for the database and ETL pipelines.
#
# DATASETS:
#   1. Student Depression Dataset (hopesb)
#      → https://www.kaggle.com/datasets/hopesb/student-depression-dataset
#   2. Student Mental Health & Resilience (ziya07)
#      → https://www.kaggle.com/datasets/ziya07/student-mental-health-and-resilience-dataset
#   3. Mental Health in Tech — OSMI (osmi)
#      → https://www.kaggle.com/datasets/osmi/mental-health-in-tech-survey
#

# OUTPUT:
#   → data/unified_dataset.csv        (the final merged clean dataset)
#   → data/unified_dataset_info.txt   (shape, dtypes, missing value report)
#   → plots/                          (distribution and correlation charts)
# =============================================================================

import os
import warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing  import LabelEncoder, MinMaxScaler
# SMOTE available via: pip install imbalanced-learn
# Fallback manual oversampling used if not installed

warnings.filterwarnings("ignore")

# ── Directory setup ────────────────────────────────────────────────────────────
os.makedirs("data",  exist_ok=True)
os.makedirs("plots", exist_ok=True)

# =============================================================================
# STEP 1 — Load the 3 raw datasets
# =============================================================================
# We read each CSV from the data/ folder.
# low_memory=False avoids pandas DtypeWarning on mixed-type columns.
# =============================================================================

print("=" * 70)
print("STEP 1 — Loading raw datasets")
print("=" * 70)

df1 = pd.read_csv("data/Student Depression Dataset.csv",  low_memory=False)
df2 = pd.read_csv("data/mental_health_dataset.csv",  low_memory=False)
df3 = pd.read_csv("data/survey.csv",  low_memory=False)

print(f"  Dataset 1 (Student Depression)  : {df1.shape[0]:>6,} rows × {df1.shape[1]} columns")
print(f"  Dataset 2 (Student Resilience)  : {df2.shape[0]:>6,} rows × {df2.shape[1]} columns")
print(f"  Dataset 3 (Mental Health Tech)  : {df3.shape[0]:>6,} rows × {df3.shape[1]} columns")

STEP 1 — Loading raw datasets
  Dataset 1 (Student Depression)  : 27,901 rows × 18 columns
  Dataset 2 (Student Resilience)  :    500 rows × 13 columns
  Dataset 3 (Mental Health Tech)  :  1,259 rows × 27 columns


In [4]:

# =============================================================================
# STEP 2 — Explore the raw data
# =============================================================================
# Before touching anything we print:
#   • Column names  — to spot naming inconsistencies across datasets
#   • Missing value counts — to decide imputation strategy
#   • Duplicate row counts — to plan deduplication
#
# This step is PURELY exploratory; nothing is modified yet.
# =============================================================================

print("\n" + "=" * 70)
print("STEP 2 — Exploratory overview")
print("=" * 70)

for label, df in [("Dataset 1", df1), ("Dataset 2", df2), ("Dataset 3", df3)]:
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    dups    = df.duplicated().sum()
    print(f"\n  {label}")
    print(f"    Columns   : {list(df.columns)}")
    print(f"    Duplicates: {dups}")
    if len(missing):
        print(f"    Missing values:\n{missing.to_string()}")
    else:
        print("    Missing values: none")


STEP 2 — Exploratory overview

  Dataset 1
    Columns   : ['id', 'Gender', 'Age', 'City', 'Profession', 'Academic Pressure', 'Work Pressure', 'CGPA', 'Study Satisfaction', 'Job Satisfaction', 'Sleep Duration', 'Dietary Habits', 'Degree', 'Have you ever had suicidal thoughts ?', 'Work/Study Hours', 'Financial Stress', 'Family History of Mental Illness', 'Depression']
    Duplicates: 0
    Missing values:
Financial Stress    3

  Dataset 2
    Columns   : ['Student_ID', 'Age', 'Gender', 'GPA', 'Stress_Level', 'Anxiety_Score', 'Depression_Score', 'Daily_Reflections', 'Sleep_Hours', 'Steps_Per_Day', 'Mood_Description', 'Sentiment_Score', 'Mental_Health_Status']
    Duplicates: 0
    Missing values: none

  Dataset 3
    Columns   : ['Timestamp', 'Age', 'Gender', 'Country', 'state', 'self_employed', 'family_history', 'treatment', 'work_interfere', 'no_employees', 'remote_work', 'tech_company', 'benefits', 'care_options', 'wellness_program', 'seek_help', 'anonymity', 'leave', 'mental_healt

In [5]:

# =============================================================================
# STEP 3 — Clean Dataset 1 (Student Depression — hopesb)
# =============================================================================
#
# This is the PRIMARY dataset — it holds our Target column: "Depression".
#
# Cleaning plan:
#   3a. Standardise column names → snake_case, strip spaces
#   3b. Drop the unnamed "id" index column if present
#   3c. Handle missing values:
#         • Numeric  → fill with median (robust to outliers)
#         • Categorical → fill with mode (most frequent value)
#   3d. Remove duplicate rows
#   3e. Normalise CGPA from 0–10 scale to standard 0–4 GPA scale
#   3f. Strip and title-case Gender to fix "male", "Male", "MALE" variants
#   3g. Encode binary Yes/No columns to 1/0 integers
#   3h. Add a source identifier column so we know where each row came from
#         after merging all three datasets
# =============================================================================

print("\n" + "=" * 70)
print("STEP 3 — Cleaning Dataset 1 (Student Depression)")
print("=" * 70)

# 3a. Standardise column names ──────────────────────────────────────────────
# Lower-case + strip whitespace + replace spaces with underscores.
# Example: "Academic Pressure" → "academic_pressure"
df1.columns = (
    df1.columns
       .str.strip()
       .str.lower()
       .str.replace(" ", "_")
       .str.replace("-", "_")
)
print("  3a. Column names standardised.")

# 3b. Drop unnamed index columns ────────────────────────────────────────────
# Kaggle CSVs often have an extra "Unnamed: 0" column from the original index.
unnamed = [c for c in df1.columns if "unnamed" in c]
if unnamed:
    df1.drop(columns=unnamed, inplace=True)
    print(f"  3b. Dropped unnamed columns: {unnamed}")
else:
    print("  3b. No unnamed columns found.")

# 3c. Handle missing values ─────────────────────────────────────────────────
# Strategy:
#   Numeric  → median imputation (safer than mean when distribution is skewed)
#   Categorical → mode imputation (most common category)
#
# We iterate over each column and check its dtype to decide the strategy.

numeric_cols     = df1.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df1.select_dtypes(include=["object"]).columns.tolist()

before_nulls = df1.isnull().sum().sum()

for col in numeric_cols:
    if df1[col].isnull().any():
        df1[col].fillna(df1[col].median(), inplace=True)

for col in categorical_cols:
    if df1[col].isnull().any():
        df1[col].fillna(df1[col].mode()[0], inplace=True)

after_nulls = df1.isnull().sum().sum()
print(f"  3c. Missing values: {before_nulls} → {after_nulls} (filled with median/mode).")

# 3d. Remove duplicates ─────────────────────────────────────────────────────
# A row is a duplicate if ALL columns match another row exactly.
# keep='first' means we keep the first occurrence and drop the rest.
before_dups = len(df1)
df1.drop_duplicates(keep="first", inplace=True)
print(f"  3d. Duplicates removed: {before_dups - len(df1)} rows dropped. Remaining: {len(df1):,}")

# 3e. Normalise CGPA from 0–10 to 0–4 scale ─────────────────────────────────
# The hopesb dataset uses a 0–10 scale. We convert to the standard 0–4
# GPA scale used internationally so it is comparable across datasets.
# Formula: cgpa_4 = cgpa_10 × (4 / 10)
if "cgpa" in df1.columns:
    df1["cgpa"] = df1["cgpa"].clip(0, 10)          # ensure no values exceed 10
    df1["cgpa"] = (df1["cgpa"] * 4 / 10).round(2)  # convert to 0–4 scale
    print("  3e. CGPA converted: 0–10 scale → 0–4 scale.")
else:
    print("  3e. CGPA column not found — skipped.")

# 3f. Fix Gender inconsistencies ────────────────────────────────────────────
# Surveys often have "male", "Male", "MALE", "m", "M" for the same value.
# We strip whitespace, title-case, then map known variants to standard labels.
if "gender" in df1.columns:
    df1["gender"] = df1["gender"].str.strip().str.title()
    gender_map = {
        "M": "Male", "F": "Female",
        "Male": "Male", "Female": "Female",
        "Man": "Male", "Woman": "Female",
    }
    df1["gender"] = df1["gender"].replace(gender_map)
    # Any remaining unknowns → 'Other'
    df1.loc[~df1["gender"].isin(["Male", "Female"]), "gender"] = "Other"
    print("  3f. Gender values standardised.")

# 3g. Encode binary Yes/No columns to 1 / 0 ─────────────────────────────────
# Machine learning models require numeric input.
# We identify columns that only contain Yes/No (or yes/no) values and map them.
# This keeps the column readable as 1=Yes, 0=No.
yes_no_cols = []
for col in df1.select_dtypes(include="object").columns:
    unique_vals = df1[col].str.strip().str.lower().dropna().unique()
    if set(unique_vals).issubset({"yes", "no"}):
        df1[col] = df1[col].str.strip().str.lower().map({"yes": 1, "no": 0})
        yes_no_cols.append(col)
print(f"  3g. Binary encoded (Yes=1, No=0): {yes_no_cols}")

# 3h. Add source identifier ─────────────────────────────────────────────────
# After merging, we need to know which dataset each row came from.
# 'Student' = from the student depression / resilience surveys
# 'Employee' = from the OSMI tech workplace survey
df1["source_type"] = "Student"
print("  3h. Added source_type = 'Student'.")

print(f"\n  ✓ Dataset 1 cleaned: {df1.shape[0]:,} rows × {df1.shape[1]} columns.")



STEP 3 — Cleaning Dataset 1 (Student Depression)
  3a. Column names standardised.
  3b. No unnamed columns found.
  3c. Missing values: 3 → 0 (filled with median/mode).
  3d. Duplicates removed: 0 rows dropped. Remaining: 27,901
  3e. CGPA converted: 0–10 scale → 0–4 scale.
  3f. Gender values standardised.
  3g. Binary encoded (Yes=1, No=0): ['have_you_ever_had_suicidal_thoughts_?', 'family_history_of_mental_illness']
  3h. Added source_type = 'Student'.

  ✓ Dataset 1 cleaned: 27,901 rows × 19 columns.


In [6]:

# =============================================================================
# STEP 4 — Clean Dataset 2 (Student Mental Health & Resilience)
# =============================================================================
#
# This dataset adds:
#   • resilience_score   — how well the student recovers from stress
#   • access_to_support  — whether support resources are available
#   • social_activities  — social engagement level
#
# These columns are NOT in Dataset 1, so they will become NEW features in
# the merged dataset (NaN for rows from other datasets — handled later).
#
# Cleaning plan: same pattern as Dataset 1.
# =============================================================================

print("\n" + "=" * 70)
print("STEP 4 — Cleaning Dataset 2 (Student Resilience)")
print("=" * 70)

# 4a. Standardise column names
df2.columns = (
    df2.columns.str.strip().str.lower()
       .str.replace(" ", "_").str.replace("-", "_")
)
print("  4a. Column names standardised.")

# 4b. Drop unnamed columns
unnamed = [c for c in df2.columns if "unnamed" in c]
if unnamed:
    df2.drop(columns=unnamed, inplace=True)
    print(f"  4b. Dropped: {unnamed}")
else:
    print("  4b. No unnamed columns.")

# 4c. Handle missing values
numeric_cols2     = df2.select_dtypes(include=[np.number]).columns
categorical_cols2 = df2.select_dtypes(include="object").columns

for col in numeric_cols2:
    df2[col].fillna(df2[col].median(), inplace=True)
for col in categorical_cols2:
    df2[col].fillna(df2[col].mode()[0], inplace=True)
print("  4c. Missing values filled.")

# 4d. Remove duplicates
before = len(df2)
df2.drop_duplicates(keep="first", inplace=True)
print(f"  4d. Duplicates removed: {before - len(df2)}. Remaining: {len(df2):,}")

# 4e. Fix Gender
if "gender" in df2.columns:
    df2["gender"] = df2["gender"].str.strip().str.title()
    df2["gender"] = df2["gender"].replace({"M": "Male", "F": "Female"})
    df2.loc[~df2["gender"].isin(["Male", "Female"]), "gender"] = "Other"
    print("  4e. Gender standardised.")

# 4f. Encode binary columns
yes_no_cols2 = []
for col in df2.select_dtypes(include="object").columns:
    unique_vals = df2[col].str.strip().str.lower().dropna().unique()
    if set(unique_vals).issubset({"yes", "no"}):
        df2[col] = df2[col].str.strip().str.lower().map({"yes": 1, "no": 0})
        yes_no_cols2.append(col)
print(f"  4f. Binary encoded: {yes_no_cols2}")

# 4g. Add source identifier
df2["source_type"] = "Student"
print("  4g. Added source_type = 'Student'.")

print(f"\n  ✓ Dataset 2 cleaned: {df2.shape[0]:,} rows × {df2.shape[1]} columns.")



STEP 4 — Cleaning Dataset 2 (Student Resilience)
  4a. Column names standardised.
  4b. No unnamed columns.
  4c. Missing values filled.
  4d. Duplicates removed: 0. Remaining: 500
  4e. Gender standardised.
  4f. Binary encoded: []
  4g. Added source_type = 'Student'.

  ✓ Dataset 2 cleaned: 500 rows × 14 columns.


In [7]:

# =============================================================================
# STEP 5 — Clean Dataset 3 (Mental Health in Tech — OSMI)
# =============================================================================
#
# This dataset covers employees, not students.
# Key differences:
#   • No CGPA or academic_pressure columns
#   • Has company-level columns: benefits, seek_help, remote_work
#   • Target is not "Depression" directly — we derive a mental_health_risk
#     binary flag from "treatment" (whether they sought treatment)
#
# Cleaning plan:
#   5a–5e. Same as above (names, unnamed, nulls, dups, gender)
#   5f.    Encode all binary Yes/No columns
#   5g.    Rename "treatment" → "depression" so it aligns with the target
#          column in Dataset 1 (a person who sought treatment = at-risk = 1)
#   5h.    Add source_type = 'Employee'
# =============================================================================

print("\n" + "=" * 70)
print("STEP 5 — Cleaning Dataset 3 (Mental Health in Tech)")
print("=" * 70)

# 5a. Standardise column names
df3.columns = (
    df3.columns.str.strip().str.lower()
       .str.replace(" ", "_").str.replace("-", "_")
       .str.replace("?", "", regex=False)
)
print("  5a. Column names standardised.")

# 5b. Drop unnamed columns
unnamed = [c for c in df3.columns if "unnamed" in c]
if unnamed:
    df3.drop(columns=unnamed, inplace=True)

# 5c. Handle missing values
# OSMI dataset uses "Don't know" and similar strings — treat them as NaN first
df3.replace(["Don't know", "Not sure", "N/A", "n/a", "NA", ""], np.nan, inplace=True)

numeric_cols3     = df3.select_dtypes(include=[np.number]).columns
categorical_cols3 = df3.select_dtypes(include="object").columns

for col in numeric_cols3:
    df3[col].fillna(df3[col].median(), inplace=True)
for col in categorical_cols3:
    if df3[col].isnull().any():
        df3[col].fillna(df3[col].mode()[0], inplace=True)
print("  5c. Missing values filled (including 'Don't know' variants).")

# 5d. Remove duplicates
before = len(df3)
df3.drop_duplicates(keep="first", inplace=True)
print(f"  5d. Duplicates removed: {before - len(df3)}. Remaining: {len(df3):,}")

# 5e. Fix Age outliers
# OSMI is a self-reported survey — some ages are clearly wrong (e.g. 3, 999).
# We keep only ages between 16 and 75.
if "age" in df3.columns:
    before_age = len(df3)
    df3 = df3[(df3["age"] >= 16) & (df3["age"] <= 75)]
    print(f"  5e. Age outliers removed: {before_age - len(df3)} rows.")

# 5f. Fix Gender — OSMI has many freeform entries ("cis male", "queer", etc.)
if "gender" in df3.columns:
    df3["gender"] = df3["gender"].str.strip().str.lower()
    def map_gender(g):
        if any(x in str(g) for x in ["female", "woman", "girl", "f "]):
            return "Female"
        if any(x in str(g) for x in ["male", "man", "boy", " m", "cis m"]):
            return "Male"
        return "Other"
    df3["gender"] = df3["gender"].apply(map_gender)
    print("  5f. Gender standardised (freeform → Male/Female/Other).")

# 5g. Encode binary Yes/No columns
yes_no_cols3 = []
for col in df3.select_dtypes(include="object").columns:
    unique_vals = df3[col].str.strip().str.lower().dropna().unique()
    if set(unique_vals).issubset({"yes", "no"}):
        df3[col] = df3[col].str.strip().str.lower().map({"yes": 1, "no": 0})
        yes_no_cols3.append(col)
print(f"  5g. Binary encoded: {yes_no_cols3}")

# 5h. Rename "treatment" → "depression" as the unified target column
# Rationale: a tech worker who sought mental health treatment is our
# proxy for "at risk" — equivalent to Depression=1 in the student datasets.
if "treatment" in df3.columns:
    df3.rename(columns={"treatment": "depression"}, inplace=True)
    print("  5h. Renamed 'treatment' → 'depression' (unified target).")

# 5i. Add source identifier
df3["source_type"] = "Employee"
print("  5i. Added source_type = 'Employee'.")

print(f"\n  ✓ Dataset 3 cleaned: {df3.shape[0]:,} rows × {df3.shape[1]} columns.")



STEP 5 — Cleaning Dataset 3 (Mental Health in Tech)
  5a. Column names standardised.
  5c. Missing values filled (including 'Don't know' variants).
  5d. Duplicates removed: 0. Remaining: 1,259
  5e. Age outliers removed: 8 rows.
  5f. Gender standardised (freeform → Male/Female/Other).
  5g. Binary encoded: ['self_employed', 'family_history', 'treatment', 'remote_work', 'tech_company', 'benefits', 'care_options', 'wellness_program', 'seek_help', 'anonymity', 'mental_vs_physical', 'obs_consequence']
  5h. Renamed 'treatment' → 'depression' (unified target).
  5i. Added source_type = 'Employee'.

  ✓ Dataset 3 cleaned: 1,251 rows × 28 columns.


In [8]:

# =============================================================================
# STEP 6 — Merge all 3 datasets into one unified DataFrame
# =============================================================================
#
# Strategy: pd.concat with outer join.
#
# Why outer?
#   Each dataset has DIFFERENT columns. Outer join keeps ALL columns from
#   ALL datasets. Where a column doesn't exist in a particular dataset,
#   it gets NaN — which we fill in the next step.
#
# Example:
#   Dataset 1 has "cgpa"          → rows from df2 & df3 get NaN for cgpa
#   Dataset 3 has "remote_work"   → rows from df1 & df2 get NaN for remote_work
#
# ignore_index=True resets the row index to 0, 1, 2, ... across all rows.
# =============================================================================

print("\n" + "=" * 70)
print("STEP 6 — Merging 3 datasets into unified DataFrame")
print("=" * 70)

unified = pd.concat([df1, df2, df3], axis=0, ignore_index=True, sort=False)

print(f"  Rows after concat    : {len(unified):,}")
print(f"  Columns after concat : {unified.shape[1]}")
print(f"  source_type counts   :\n{unified['source_type'].value_counts().to_string()}")



STEP 6 — Merging 3 datasets into unified DataFrame
  Rows after concat    : 29,652
  Columns after concat : 54
  source_type counts   :
source_type
Student     28401
Employee     1251


In [9]:
# =============================================================================
# STEP 7 — Post-merge missing value handling
# =============================================================================
#
# After the outer join, many columns will have NaN for rows where that column
# didn't exist in the source dataset.
#
# Strategy:
#   • Numeric cross-dataset NaN  → fill with the GLOBAL median of that column
#     (calculated only from rows that DO have the value — i.e., no bias)
#   • Categorical cross-dataset NaN → fill with "Unknown" to be honest about
#     the fact that we don't have this information for that row
#
# We do NOT impute with 0 for numeric because 0 often has a specific meaning
# (e.g., 0 financial stress = no stress at all, which is different from
# "we don't know their financial stress level").
# =============================================================================
 
print("\n" + "=" * 70)
print("STEP 7 — Post-merge missing value handling")
print("=" * 70)
 
total_nulls_before = unified.isnull().sum().sum()
 
numeric_unified     = unified.select_dtypes(include=[np.number]).columns
categorical_unified = unified.select_dtypes(include="object").columns
 
for col in numeric_unified:
    if unified[col].isnull().any():
        unified[col].fillna(unified[col].median(), inplace=True)
 
for col in categorical_unified:
    if col in ["source_type"]:
        continue  # already set
    if unified[col].isnull().any():
        unified[col].fillna("Unknown", inplace=True)
 
total_nulls_after = unified.isnull().sum().sum()
print(f"  Total missing values: {total_nulls_before:,} → {total_nulls_after:,}")
 


STEP 7 — Post-merge missing value handling
  Total missing values: 1,029,061 → 0


In [10]:

# =============================================================================
# STEP 8 — Normalise numeric features to 0–1 range
# =============================================================================
#
# Why normalise?
#   ML models (especially distance-based and gradient-based ones) are sensitive
#   to feature scale. A column ranging 0–10 would dominate a column ranging
#   0–1 even if both are equally important.
#
# We use Min-Max Scaling: x_scaled = (x - x_min) / (x_max - x_min)
#   → All values land in [0, 1]
#   → Original distribution shape is preserved
#
# We scale:
#   academic_pressure, work_pressure, financial_stress, cgpa,
#   study_satisfaction, job_satisfaction, stress_level, resilience_score
#   (all the 1–5 and 1–10 scale columns)
#
# We do NOT scale binary columns (already 0/1) or the target "depression".
# =============================================================================
 
print("\n" + "=" * 70)
print("STEP 8 — Min-Max normalisation of numeric features")
print("=" * 70)
 
# Columns to scale — only include those that exist in the unified DataFrame
scale_candidates = [
    "academic_pressure", "work_pressure", "financial_stress",
    "cgpa", "study_satisfaction", "job_satisfaction",
    "work_study_hours", "stress_level", "resilience_score",
    "social_activities", "age"
]
cols_to_scale = [c for c in scale_candidates if c in unified.columns]
 
scaler = MinMaxScaler()
unified[cols_to_scale] = scaler.fit_transform(unified[cols_to_scale])
 
print(f"  Scaled columns: {cols_to_scale}")
 


STEP 8 — Min-Max normalisation of numeric features
  Scaled columns: ['academic_pressure', 'work_pressure', 'financial_stress', 'cgpa', 'study_satisfaction', 'job_satisfaction', 'stress_level', 'age']


In [11]:

# =============================================================================
# STEP 9 — Encode remaining categorical columns
# =============================================================================
#
# After binary encoding in Steps 3–5, some multi-class categorical columns
# still hold string values (e.g., "city", "degree", "country",
# "sleep_duration", "dietary_habits", "mental_health_status").
#
# Strategy: Label Encoding
#   Each unique string is replaced with an integer.
#   Example: "Bachelors"=0, "Masters"=1, "PhD"=2
#
# Note: For tree-based models (Random Forest) label encoding is fine.
#       For linear models, one-hot encoding would be preferred — but we
#       keep it simple here and note it in comments.
#
# We skip: "source_type" (used for tracking, not training)
#          "depression"  (this is the TARGET — don't encode it)
# =============================================================================
 
print("\n" + "=" * 70)
print("STEP 9 — Label encoding remaining categorical columns")
print("=" * 70)
 
skip_cols = {"source_type", "depression"}
le = LabelEncoder()
encoded_cols = []
 
for col in unified.select_dtypes(include="object").columns:
    if col in skip_cols:
        continue
    unified[col] = le.fit_transform(unified[col].astype(str))
    encoded_cols.append(col)
 
print(f"  Label-encoded: {encoded_cols}")
 


STEP 9 — Label encoding remaining categorical columns
  Label-encoded: ['gender', 'city', 'profession', 'sleep_duration', 'dietary_habits', 'degree', 'daily_reflections', 'mood_description', 'timestamp', 'country', 'state', 'work_interfere', 'no_employees', 'leave', 'mental_health_consequence', 'phys_health_consequence', 'coworkers', 'supervisor', 'mental_health_interview', 'phys_health_interview', 'comments']


In [12]:
 
# =============================================================================
# STEP 10 — Ensure the target column "depression" is numeric (0 / 1)
# =============================================================================
#
# The target column may still be a string if not caught in the Yes/No pass.
# We unify: 1 = has depression / is at risk, 0 = not at risk.
# =============================================================================
 
print("\n" + "=" * 70)
print("STEP 10 — Validate and finalise the target column 'depression'")
print("=" * 70)
 
if "depression" in unified.columns:
    # Handle string variants that weren't caught earlier
    if unified["depression"].dtype == object:
        unified["depression"] = (
            unified["depression"]
            .str.strip().str.lower()
            .map({"yes": 1, "no": 0, "1": 1, "0": 0})
        )
 
    # Convert to integer
    unified["depression"] = unified["depression"].fillna(0).astype(int)
    dist = unified["depression"].value_counts()
    print(f"  Target distribution:\n    0 (No depression) : {dist.get(0, 0):,}\n    1 (Depression)    : {dist.get(1, 0):,}")
    pct = dist.get(1, 0) / len(unified) * 100
    print(f"  Positive class rate: {pct:.1f}%")
else:
    print("  WARNING: 'depression' column not found in unified DataFrame!")
 


STEP 10 — Validate and finalise the target column 'depression'
  Target distribution:
    0 (No depression) : 12,184
    1 (Depression)    : 17,468
  Positive class rate: 58.9%


In [13]:

# =============================================================================
# STEP 11 — Handle class imbalance with SMOTE
# =============================================================================
#
# What is class imbalance?
#   In mental health datasets, depressed students are usually a MINORITY
#   (e.g., 30% of records). An ML model trained on imbalanced data will
#   learn to predict "No Depression" for everything and still get 70% accuracy
#   — which is useless in practice.
#
# What is SMOTE?
#   Synthetic Minority Over-sampling TEchnique.
#   Instead of just duplicating minority rows, SMOTE creates NEW synthetic
#   samples by interpolating between existing minority-class rows.
#   This produces a more diverse set of minority examples.
#
# We apply SMOTE on all numeric features, then reconstruct the full DataFrame.
# =============================================================================
 
print("\n" + "=" * 70)
print("STEP 11 — SMOTE oversampling to balance classes")
print("=" * 70)
 
if "depression" in unified.columns:
    # Separate features and target
    # We use all numeric columns except the target itself and source_type
    feature_cols = [
        c for c in unified.select_dtypes(include=[np.number]).columns
        if c not in ["depression"]
    ]
    X = unified[feature_cols].values
    y = unified["depression"].values
 
    # Apply SMOTE
    try:
        from imblearn.over_sampling import SMOTE
        sm = SMOTE(random_state=42)
        X_res, y_res = sm.fit_resample(X, y)
    except ImportError:
        from sklearn.utils import resample
        minority_idx = np.where(y == 1)[0]
        majority_idx = np.where(y == 0)[0]
        minority_up  = resample(minority_idx, replace=True,
                                n_samples=len(majority_idx), random_state=42)
        idx_balanced = np.concatenate([majority_idx, minority_up])
        X_res = X[idx_balanced]
        y_res = y[idx_balanced]
 
    # Reconstruct DataFrame
    unified_balanced = pd.DataFrame(X_res, columns=feature_cols)
    unified_balanced["depression"] = y_res
 
    dist_after = pd.Series(y_res).value_counts()
    print(f"  After SMOTE:")
    print(f"    0 (No depression) : {dist_after.get(0, 0):,}")
    print(f"    1 (Depression)    : {dist_after.get(1, 0):,}")
    print(f"  Total rows after SMOTE: {len(unified_balanced):,}")
else:
    unified_balanced = unified.copy()
    print("  Skipped — 'depression' column not found.")
 


STEP 11 — SMOTE oversampling to balance classes
  After SMOTE:
    0 (No depression) : 17,468
    1 (Depression)    : 17,468
  Total rows after SMOTE: 34,936


In [14]:

# =============================================================================
# STEP 12 — Remove outliers using IQR filtering
# =============================================================================
#
# What is IQR (Interquartile Range)?
#   IQR = Q3 - Q1  (75th percentile minus 25th percentile)
#   Any value below Q1 - 1.5×IQR or above Q3 + 1.5×IQR is an outlier.
#
# Why do this AFTER SMOTE?
#   SMOTE creates synthetic points by interpolation — they shouldn't create
#   outliers. But original extreme values could still skew the feature space.
#   We remove them after SMOTE so we don't remove real minority-class points
#   before oversampling (that would make the imbalance worse).
#
# We only apply IQR to the original pressure / score columns (post-scale),
# not to binary 0/1 columns.
# =============================================================================
 
print("\n" + "=" * 70)
print("STEP 12 — IQR-based outlier removal")
print("=" * 70)
 
before_outlier = len(unified_balanced)
numeric_for_iqr = [c for c in cols_to_scale if c in unified_balanced.columns]
 
# We apply IQR per column independently and flag outlier ROWS.
# A row is only dropped if it is an outlier in MORE THAN HALF of the
# scaled feature columns — this avoids over-aggressive filtering that
# would remove valid rows just because one feature is at its natural extreme.
 
outlier_flags = pd.DataFrame(index=unified_balanced.index)
 
for col in numeric_for_iqr:
    Q1 = unified_balanced[col].quantile(0.25)
    Q3 = unified_balanced[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outlier_flags[col] = ~((unified_balanced[col] >= lower) & (unified_balanced[col] <= upper))
 
# Drop only rows that are outliers in more than half the checked columns
threshold   = len(numeric_for_iqr) // 2
is_outlier  = outlier_flags.sum(axis=1) > threshold
unified_balanced = unified_balanced[~is_outlier].reset_index(drop=True)
print(f"  Rows before IQR filter: {before_outlier:,}")
print(f"  Rows after  IQR filter: {len(unified_balanced):,}")
print(f"  Removed: {before_outlier - len(unified_balanced):,} outlier rows")
 


STEP 12 — IQR-based outlier removal
  Rows before IQR filter: 34,936
  Rows after  IQR filter: 34,936
  Removed: 0 outlier rows


In [15]:

# =============================================================================
# STEP 13 — Final validation and report
# =============================================================================
 
print("\n" + "=" * 70)
print("STEP 13 — Final validation")
print("=" * 70)
 
final_nulls = unified_balanced.isnull().sum().sum()
print(f"  Final shape        : {unified_balanced.shape}")
print(f"  Total null values  : {final_nulls}")
print(f"  Dtypes breakdown   :\n{unified_balanced.dtypes.value_counts().to_string()}")
 
if "depression" in unified_balanced.columns:
    final_dist = unified_balanced["depression"].value_counts()
    print(f"\n  Final target distribution:")
    print(f"    0 (No depression): {final_dist.get(0, 0):,}")
    print(f"    1 (Depression)   : {final_dist.get(1, 0):,}")
 


STEP 13 — Final validation
  Final shape        : (34936, 53)
  Total null values  : 0
  Dtypes breakdown   :
float64    52
int32       1

  Final target distribution:
    0 (No depression): 17,468
    1 (Depression)   : 17,468


In [16]:

# =============================================================================
# STEP 14 — Visualisations
# =============================================================================
#
# We generate 4 charts saved to the plots/ folder:
#   1. Depression class distribution (before vs. after SMOTE)
#   2. Correlation heatmap of numeric features
#   3. Age distribution by depression status
#   4. Financial stress distribution by depression status
# =============================================================================
 
print("\n" + "=" * 70)
print("STEP 14 — Generating visualisation plots")
print("=" * 70)
 
sns.set_theme(style="whitegrid", palette="muted")
 
# Chart 1: Target class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
 
before_dist = unified["depression"].value_counts() if "depression" in unified.columns else pd.Series()
after_dist  = unified_balanced["depression"].value_counts() if "depression" in unified_balanced.columns else pd.Series()
 
if not before_dist.empty:
    axes[0].bar(["No Depression (0)", "Depression (1)"],
                [before_dist.get(0, 0), before_dist.get(1, 0)],
                color=["#2196F3", "#F44336"])
    axes[0].set_title("Before SMOTE — Class Distribution", fontsize=13, fontweight="bold")
    axes[0].set_ylabel("Count")
 
if not after_dist.empty:
    axes[1].bar(["No Depression (0)", "Depression (1)"],
                [after_dist.get(0, 0), after_dist.get(1, 0)],
                color=["#2196F3", "#F44336"])
    axes[1].set_title("After SMOTE — Class Distribution", fontsize=13, fontweight="bold")
    axes[1].set_ylabel("Count")
 
plt.tight_layout()
plt.savefig("plots/01_class_distribution.png", dpi=150)
plt.close()
print("  Saved: plots/01_class_distribution.png")
 
# Chart 2: Correlation heatmap
numeric_plot_cols = [c for c in unified_balanced.select_dtypes(include=np.number).columns
                     if c != "depression"][:12]  # limit to 12 for readability
if len(numeric_plot_cols) >= 2:
    corr = unified_balanced[numeric_plot_cols + ["depression"]].corr()
    fig, ax = plt.subplots(figsize=(14, 10))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
                center=0, linewidths=0.5, ax=ax,
                annot_kws={"size": 8})
    ax.set_title("Feature Correlation Heatmap", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig("plots/02_correlation_heatmap.png", dpi=150)
    plt.close()
    print("  Saved: plots/02_correlation_heatmap.png")
 
# Chart 3: Age distribution by depression status
if "age" in unified_balanced.columns and "depression" in unified_balanced.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    for label, color in [(0, "#2196F3"), (1, "#F44336")]:
        subset = unified_balanced[unified_balanced["depression"] == label]["age"]
        ax.hist(subset, bins=30, alpha=0.6, color=color,
                label=f"{'No Depression' if label == 0 else 'Depression'}")
    ax.set_title("Age Distribution by Depression Status", fontsize=13, fontweight="bold")
    ax.set_xlabel("Age (normalised 0–1)")
    ax.set_ylabel("Count")
    ax.legend()
    plt.tight_layout()
    plt.savefig("plots/03_age_distribution.png", dpi=150)
    plt.close()
    print("  Saved: plots/03_age_distribution.png")
 
# Chart 4: Financial stress vs. depression
if "financial_stress" in unified_balanced.columns and "depression" in unified_balanced.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    for label, color in [(0, "#2196F3"), (1, "#F44336")]:
        subset = unified_balanced[unified_balanced["depression"] == label]["financial_stress"]
        ax.hist(subset, bins=25, alpha=0.6, color=color,
                label=f"{'No Depression' if label == 0 else 'Depression'}")
    ax.set_title("Financial Stress Distribution by Depression Status", fontsize=13, fontweight="bold")
    ax.set_xlabel("Financial Stress (normalised 0–1)")
    ax.set_ylabel("Count")
    ax.legend()
    plt.tight_layout()
    plt.savefig("plots/04_financial_stress_distribution.png", dpi=150)
    plt.close()
    print("  Saved: plots/04_financial_stress_distribution.png")
 


STEP 14 — Generating visualisation plots
  Saved: plots/01_class_distribution.png
  Saved: plots/02_correlation_heatmap.png
  Saved: plots/03_age_distribution.png
  Saved: plots/04_financial_stress_distribution.png


In [17]:

# =============================================================================
# STEP 15 — Save the final unified dataset and info report
# =============================================================================
 
print("\n" + "=" * 70)
print("STEP 15 — Saving output files")
print("=" * 70)
 
# Save the main CSV
output_csv = "data/unified_dataset.csv"
unified_balanced.to_csv(output_csv, index=False)
print(f"  ✓ Saved: {output_csv}  ({len(unified_balanced):,} rows)")
 
# Save a text report with shape, dtypes, and missing value summary
report_path = "data/unified_dataset_info.txt"
with open(report_path, "w") as f:
    f.write("MindGuard — Milestone 1 Dataset Report\n")
    f.write("=" * 50 + "\n\n")
    f.write(f"Final shape: {unified_balanced.shape}\n\n")
    f.write("Column dtypes:\n")
    f.write(unified_balanced.dtypes.to_string())
    f.write("\n\nMissing values per column:\n")
    f.write(unified_balanced.isnull().sum().to_string())
    f.write("\n\nBasic statistics:\n")
    f.write(unified_balanced.describe().to_string())
 
print(f"  ✓ Saved: {report_path}")
 
# =============================================================================
# DONE
# =============================================================================
 
print("\n" + "=" * 70)
print("  Milestone 1 complete!")
print(f"  Final dataset: {unified_balanced.shape[0]:,} rows × {unified_balanced.shape[1]} columns")
print("  Ready for Milestone 2 — Database Design & Integration")
print("=" * 70)


STEP 15 — Saving output files
  ✓ Saved: data/unified_dataset.csv  (34,936 rows)
  ✓ Saved: data/unified_dataset_info.txt

  Milestone 1 complete!
  Final dataset: 34,936 rows × 53 columns
  Ready for Milestone 2 — Database Design & Integration
